# Corrigé du TP3 : construction d'indicateurs consolidés

## Objectifs

- parcourir automatiquement un dossier de fichiers avec `pathlib` et `glob`
- assembler plusieurs fichiers en une seule table avec `pd.concat`, et contrôler le résultat
- comprendre les types de jointure, puis enrichir les ventes avec des tables de référence grâce à `merge`
- calculer des indicateurs avec `groupby`, `agg`, `transform`, `pivot_table` et `crosstab`
- manipuler les dates : composantes, filtres de période, `resample`, évolution d'un mois sur l'autre, moyenne mobile
- produire et exporter une table d'indicateurs par mois et par région

## Livrable

- une table d'indicateurs `indicateurs_mois_region.parquet`, avec une ligne par mois et par région, prête pour le tableau de bord de cet après-midi
- la table détaillée des ventes enrichies, `ventes_consolidees.parquet`
- trois ou quatre constats tirés des indicateurs, dans la synthèse en fin de notebook

## Les données

```text
ressources/
  ventes/             un fichier de ventes par mois, de janvier à septembre 2025
  produits.csv        le catalogue : id_produit, produit, categorie, prix_catalogue
  regions.csv         les régions : code_region, region, objectif_mensuel
```

## Comment travailler

- Le TP suit le cours : chaque partie se fait juste après la section du cours correspondante.
- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués dans l'énoncé.
- Les questions s'enchaînent : les tables construites au début (`fichiers`, `ventes`, `df`) servent jusqu'à la fin. Si tu rouvres le notebook, réexécute d'abord les cellules précédentes.
- Quand un **résultat attendu** est indiqué, compare-le avec ce que tu obtiens.
- Les cellules qui commencent par `# Vérification` contrôlent ton travail avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK` ; sinon, elles s'arrêtent sur une `AssertionError` dont le message indique ce qui ne va pas.
- Les questions **Pour aller plus loin** sont facultatives : fais-les si tu as terminé une partie en avance.

## Partie 0 : environnement

*À faire après la section 1 du cours : les données du jour.*

Comme au TP2, deux possibilités :

- réutiliser l'environnement du TP1 : active-le, puis lance `jupyter lab` depuis le dossier de ce TP ;
- ou recréer un environnement dans ce dossier à partir du `requirements.txt` du TP1 :
  ```bash
  python3 -m venv .venv
  source .venv/bin/activate
  python -m pip install -r ../tp01-environnement-pandas/requirements.txt
  jupyter lab
  ```

Ouvre ensuite le dossier `ressources` dans l'explorateur de fichiers de JupyterLab, et regarde le contenu d'un fichier de ventes et des deux tables de référence. Exécute enfin la cellule suivante.

In [1]:
from pathlib import Path

import pandas as pd

DOSSIER = Path("ressources")


## Partie 1 : parcourir un dossier et empiler les fichiers

*À faire après la section 2 du cours : parcourir un dossier et empiler des fichiers.*

### Question 1.1

1. Construis avec l'opérateur `/`, à partir de `DOSSIER`, le chemin `fichier_janvier` du fichier `ventes_2025_01.csv`, rangé dans le sous-dossier `ventes`.
2. Affiche son nom (`name`), son nom sans extension (`stem`), son extension (`suffix`), son dossier parent (`parent`), et vérifie qu'il existe (`exists()`).
3. Le nom du fichier contient le numéro du mois. Récupère-le à partir de `stem` dans une variable `mois_janvier`, sous forme d'entier.

Indice : `"a_b_c".split("_")` découpe un texte en une liste `["a", "b", "c"]`, et `int("01")` vaut `1`.

**Résultat attendu** :

```text
ventes_2025_01.csv
ventes_2025_01
.csv
ressources/ventes
True
1
```

In [2]:
fichier_janvier = DOSSIER / "ventes" / "ventes_2025_01.csv"
print(fichier_janvier.name)
print(fichier_janvier.stem)
print(fichier_janvier.suffix)
print(fichier_janvier.parent)
print(fichier_janvier.exists())

mois_janvier = int(fichier_janvier.stem.split("_")[-1])
print(mois_janvier)


ventes_2025_01.csv
ventes_2025_01
.csv
ressources/ventes
True
1


In [3]:
# Vérification
assert fichier_janvier == Path("ressources/ventes/ventes_2025_01.csv"), "construis le chemin avec DOSSIER / \"ventes\" / \"ventes_2025_01.csv\""
assert fichier_janvier.exists(), "le fichier est introuvable : Jupyter a-t-il été lancé depuis le dossier du TP ?"
assert mois_janvier == 1 and isinstance(mois_janvier, int), "mois_janvier doit être l'entier 1"
print("Question 1.1 : OK")


Question 1.1 : OK


### Question 1.2

Liste les fichiers CSV du dossier `ressources/ventes` avec `glob`, triés dans l'ordre alphabétique, dans une variable `fichiers`. Affiche leur nombre et leurs noms.

**Résultat attendu** :

```text
9 fichiers
['ventes_2025_01.csv',
 'ventes_2025_02.csv',
 'ventes_2025_03.csv',
 'ventes_2025_04.csv',
 'ventes_2025_05.csv',
 'ventes_2025_06.csv',
 'ventes_2025_07.csv',
 'ventes_2025_08.csv',
 'ventes_2025_09.csv']
```

In [4]:
fichiers = sorted((DOSSIER / "ventes").glob("*.csv"))
print(len(fichiers), "fichiers")
[f.name for f in fichiers]


9 fichiers


['ventes_2025_01.csv',
 'ventes_2025_02.csv',
 'ventes_2025_03.csv',
 'ventes_2025_04.csv',
 'ventes_2025_05.csv',
 'ventes_2025_06.csv',
 'ventes_2025_07.csv',
 'ventes_2025_08.csv',
 'ventes_2025_09.csv']

In [5]:
# Vérification
assert len(fichiers) == 9, "le dossier ressources/ventes contient 9 fichiers CSV"
assert isinstance(fichiers[0], Path), "fichiers doit contenir les objets Path renvoyés par glob, pas des noms"
assert [f.name for f in fichiers] == [f"ventes_2025_{m:02d}.csv" for m in range(1, 10)], "trie la liste avec sorted"
print("Question 1.2 : OK")


Question 1.2 : OK


### Question 1.3

Essaie d'autres patterns, en triant chaque résultat avec `sorted` :

1. `premier_trimestre` : les fichiers de janvier à mars, avec le pattern `ventes_2025_0[1-3].csv` ;
2. `ete` : les fichiers de juin à août ;
3. `tous_les_csv` : tous les fichiers CSV du dossier `ressources`, sous-dossiers compris, avec `rglob`.

Affiche les noms des fichiers des deux premières listes, puis le nombre de fichiers de la troisième. Pourquoi en trouve-t-on 11 ?

**Résultat attendu** :

```text
['ventes_2025_01.csv', 'ventes_2025_02.csv', 'ventes_2025_03.csv']
['ventes_2025_06.csv', 'ventes_2025_07.csv', 'ventes_2025_08.csv']
11
```

In [6]:
premier_trimestre = sorted((DOSSIER / "ventes").glob("ventes_2025_0[1-3].csv"))
ete = sorted((DOSSIER / "ventes").glob("ventes_2025_0[6-8].csv"))
tous_les_csv = sorted(DOSSIER.rglob("*.csv"))

print([f.name for f in premier_trimestre])
print([f.name for f in ete])
print(len(tous_les_csv))


['ventes_2025_01.csv', 'ventes_2025_02.csv', 'ventes_2025_03.csv']
['ventes_2025_06.csv', 'ventes_2025_07.csv', 'ventes_2025_08.csv']
11


In [7]:
# Vérification
assert [f.name for f in premier_trimestre] == ["ventes_2025_01.csv", "ventes_2025_02.csv", "ventes_2025_03.csv"], "premier_trimestre : pattern ventes_2025_0[1-3].csv"
assert [f.name for f in ete] == ["ventes_2025_06.csv", "ventes_2025_07.csv", "ventes_2025_08.csv"], "ete : pattern ventes_2025_0[6-8].csv"
assert len(tous_les_csv) == 11, "rglob parcourt aussi les sous-dossiers"
print("Question 1.3 : OK")


Question 1.3 : OK


**Réponse** : `rglob` parcourt le dossier `ressources` et tous ses sous-dossiers : il trouve les 9 fichiers de ventes, plus `produits.csv` et `regions.csv`.

### Question 1.4

Avant d'assembler les fichiers, compte les lignes de chacun : construis avec une compréhension le dictionnaire `lignes_par_fichier`, qui associe le nom de chaque fichier à son nombre de lignes (`len` d'un DataFrame donne son nombre de lignes). Affiche chaque nom et son nombre de lignes avec une boucle, puis le total. Quel mois compte le moins de ventes ?

**Résultat attendu** :

```text
ventes_2025_01.csv 226
ventes_2025_02.csv 229
ventes_2025_03.csv 286
ventes_2025_04.csv 260
ventes_2025_05.csv 259
ventes_2025_06.csv 296
ventes_2025_07.csv 171
ventes_2025_08.csv 136
ventes_2025_09.csv 319
total : 2182
```

In [8]:
lignes_par_fichier = {f.name: len(pd.read_csv(f)) for f in fichiers}

for nom, nb in lignes_par_fichier.items():
    print(nom, nb)
print("total :", sum(lignes_par_fichier.values()))


ventes_2025_01.csv 226
ventes_2025_02.csv 229
ventes_2025_03.csv 286
ventes_2025_04.csv 260
ventes_2025_05.csv 259
ventes_2025_06.csv 296
ventes_2025_07.csv 171
ventes_2025_08.csv 136
ventes_2025_09.csv 319
total : 2182


In [9]:
# Vérification
assert len(lignes_par_fichier) == 9, "une entrée par fichier"
assert lignes_par_fichier["ventes_2025_08.csv"] == 136, "les clés sont les noms des fichiers (f.name), les valeurs leur nombre de lignes"
assert sum(lignes_par_fichier.values()) == 2182, "le total doit valoir 2182 lignes"
print("Question 1.4 : OK")


Question 1.4 : OK


**Réponse** : Août, avec 136 ventes, suivi de juillet avec 171 : l'activité baisse nettement pendant l'été.

### Question 1.5

Lis le premier fichier et le fichier de mai (`fichiers[4]`), et affiche la liste de leurs colonnes. Que remarques-tu ? Est-ce un problème pour les assembler ?

**Résultat attendu** :

```text
['id_commande', 'date', 'code_region', 'id_produit', 'quantite', 'prix_unitaire', 'client_fidele']
['date', 'id_commande', 'id_produit', 'code_region', 'prix_unitaire', 'quantite', 'client_fidele']
```

In [10]:
print(list(pd.read_csv(fichiers[0]).columns))
print(list(pd.read_csv(fichiers[4]).columns))


['id_commande', 'date', 'code_region', 'id_produit', 'quantite', 'prix_unitaire', 'client_fidele']
['date', 'id_commande', 'id_produit', 'code_region', 'prix_unitaire', 'quantite', 'client_fidele']


**Réponse** : Les deux fichiers ont les mêmes colonnes, mais pas dans le même ordre. Ce n'est pas un problème : `concat` aligne les colonnes par leur nom, pas par leur position.

### Question 1.6

Écris une fonction `charger_ventes(fichier)` qui lit un fichier de ventes en convertissant la colonne `date`, et ajoute une colonne `fichier` contenant le nom du fichier d'origine (sans le dossier).

Teste-la sur le premier fichier, dans une variable `essai` : affiche sa forme et les types de ses colonnes.

**Résultat attendu** :

```text
(226, 8)
id_commande               int64
date             datetime64[us]
code_region                 str
id_produit                  str
quantite                  int64
prix_unitaire           float64
client_fidele              bool
fichier                     str
dtype: object
```

In [11]:
def charger_ventes(fichier):
    morceau = pd.read_csv(fichier, parse_dates=["date"])
    morceau["fichier"] = fichier.name
    return morceau


essai = charger_ventes(fichiers[0])
print(essai.shape)
essai.dtypes


(226, 8)


id_commande               int64
date             datetime64[us]
code_region                 str
id_produit                  str
quantite                  int64
prix_unitaire           float64
client_fidele              bool
fichier                     str
dtype: object

In [12]:
# Vérification
essai_mai = charger_ventes(fichiers[4])
assert essai_mai.shape == (259, 8), "la fonction doit renvoyer les 7 colonnes du fichier, plus la colonne fichier"
assert pd.api.types.is_datetime64_any_dtype(essai_mai["date"]), "la colonne date doit être convertie : parse_dates"
assert (essai_mai["fichier"] == "ventes_2025_05.csv").all(), "la colonne fichier doit contenir le nom du fichier, sans le dossier : fichier.name"
print("Question 1.6 : OK")


Question 1.6 : OK


### Question 1.7

Assemble tous les fichiers dans un DataFrame `ventes`, avec un seul appel à `pd.concat` et une compréhension de liste. Affiche sa forme.

**Résultat attendu** :

```text
(2182, 8)
```

In [13]:
ventes = pd.concat([charger_ventes(f) for f in fichiers], ignore_index=True)
ventes.shape


(2182, 8)

In [14]:
# Vérification
assert ventes.shape == (2182, 8), "ventes doit contenir 2182 lignes et 8 colonnes"
assert ventes.index.equals(pd.RangeIndex(2182)), "les lignes doivent être renumérotées de 0 à 2181 : ignore_index=True"
assert list(ventes.columns[:2]) == ["id_commande", "date"], "les colonnes suivent l'ordre du premier fichier"
print("Question 1.7 : OK")


Question 1.7 : OK


### Question 1.8

Assemble à nouveau les fichiers, mais sans `ignore_index=True`, dans une variable `sans_ignore`. Affiche si son index est unique (`sans_ignore.index.is_unique`), puis le nombre de lignes renvoyées par `sans_ignore.loc[0]`. Explique.

**Résultat attendu** :

```text
False
9
```

In [15]:
sans_ignore = pd.concat([charger_ventes(f) for f in fichiers])
print(sans_ignore.index.is_unique)
print(len(sans_ignore.loc[0]))


False
9


**Réponse** : Chaque fichier apporte son propre index, qui commence à 0 : le label 0 existe une fois par fichier, et `loc[0]` renvoie 9 lignes. Avec `ignore_index=True`, les lignes sont renumérotées de 0 à 2181 et chaque label désigne une seule ligne.

### Question 1.9

Contrôle le résultat de l'assemblage :

1. le nombre de lignes par fichier, dans une variable `controle_fichiers` (`value_counts`, puis `sort_index`) ;
2. la liste des colonnes ;
3. le nombre total de valeurs manquantes ;
4. l'unicité de `id_commande`.

La vérification compare `controle_fichiers` aux comptes de la question 1.4.

**Résultat attendu** :

```text
fichier
ventes_2025_01.csv    226
ventes_2025_02.csv    229
ventes_2025_03.csv    286
ventes_2025_04.csv    260
ventes_2025_05.csv    259
ventes_2025_06.csv    296
ventes_2025_07.csv    171
ventes_2025_08.csv    136
ventes_2025_09.csv    319
Name: count, dtype: int64
['id_commande', 'date', 'code_region', 'id_produit', 'quantite', 'prix_unitaire', 'client_fidele', 'fichier']
0
True
```

In [16]:
controle_fichiers = ventes["fichier"].value_counts().sort_index()
print(controle_fichiers)
print(list(ventes.columns))
print(ventes.isna().sum().sum())
print(ventes["id_commande"].is_unique)


fichier
ventes_2025_01.csv    226
ventes_2025_02.csv    229
ventes_2025_03.csv    286
ventes_2025_04.csv    260
ventes_2025_05.csv    259
ventes_2025_06.csv    296
ventes_2025_07.csv    171
ventes_2025_08.csv    136
ventes_2025_09.csv    319
Name: count, dtype: int64
['id_commande', 'date', 'code_region', 'id_produit', 'quantite', 'prix_unitaire', 'client_fidele', 'fichier']
0
True


In [17]:
# Vérification
assert controle_fichiers.to_dict() == lignes_par_fichier, "le nombre de lignes par fichier doit être le même qu'avant l'assemblage"
assert ventes.isna().sum().sum() == 0, "aucune valeur ne doit manquer après l'assemblage"
assert ventes["id_commande"].is_unique, "aucune commande ne doit apparaître dans deux fichiers"
print("Question 1.9 : OK")


Question 1.9 : OK


### Pour aller plus loin 1.10

Assemble les fichiers en une seule expression, sans la fonction `charger_ventes` : une compréhension de liste qui lit chaque fichier et lui ajoute la colonne `fichier` avec `assign`. Range le résultat dans `ventes_bis`, puis vérifie avec la méthode `equals` qu'il est identique à `ventes`.

**Résultat attendu** :

```text
True
```

In [18]:
ventes_bis = pd.concat(
    [pd.read_csv(f, parse_dates=["date"]).assign(fichier=f.name) for f in fichiers],
    ignore_index=True,
)
ventes_bis.equals(ventes)


True

In [19]:
# Vérification
assert ventes_bis.equals(ventes), "ventes_bis doit être identique à ventes : mêmes colonnes, mêmes valeurs, même index"
print("Question 1.10 : OK")


Question 1.10 : OK


### Pour aller plus loin 1.11

Deux exports d'octobre doivent être empilés, mais le second nomme sa colonne de date `Date`, avec une majuscule. Empile-les une première fois tels quels et affiche le résultat : que se passe-t-il ? Corrige ensuite le nom de la colonne avec `rename` avant d'empiler, dans une variable `octobre`.

**Résultat attendu** :

```text
         date  montant        Date
0  2025-10-01    120.0         NaN
1  2025-10-02     80.0         NaN
2         NaN     45.0  2025-10-03
         date  montant
0  2025-10-01    120.0
1  2025-10-02     80.0
2  2025-10-03     45.0
```

In [20]:
export_a = pd.DataFrame({"date": ["2025-10-01", "2025-10-02"], "montant": [120.0, 80.0]})
export_b = pd.DataFrame({"Date": ["2025-10-03"], "montant": [45.0]})

print(pd.concat([export_a, export_b], ignore_index=True))

octobre = pd.concat([export_a, export_b.rename(columns={"Date": "date"})], ignore_index=True)
octobre


         date  montant        Date
0  2025-10-01    120.0         NaN
1  2025-10-02     80.0         NaN
2         NaN     45.0  2025-10-03


,date,montant
0,2025-10-01,120.0
1,2025-10-02,80.0
2,2025-10-03,45.0


In [21]:
# Vérification
assert list(octobre.columns) == ["date", "montant"], "octobre ne doit avoir que les colonnes date et montant"
assert octobre["date"].notna().all() and len(octobre) == 3, "les trois lignes doivent avoir une date"
print("Question 1.11 : OK")


Question 1.11 : OK


**Réponse** : `concat` aligne les colonnes par leur nom : `date` et `Date` sont deux colonnes différentes. Le résultat en contient trois, et chaque colonne de date est vide (`NaN`) pour les lignes de l'autre fichier. Aucune erreur n'est signalée : seul un contrôle des colonnes révèle le problème.

## Partie 2 : joindre des tables

*À faire après la section 3 du cours : joindre des tables.*

On commence par de très petites tables, pour voir ligne par ligne l'effet de chaque type de jointure, puis on passe aux vraies données.

### Question 2.1

Les deux tables ci-dessous représentent des commandes et un fichier clients. La commande 4 concerne un client `C4` absent du fichier clients, et le client `C5` n'a passé aucune commande.

Avant d'exécuter quoi que ce soit, prévois le nombre de lignes de chaque jointure. Joins ensuite `commandes` et `clients` sur `code_client` de trois façons, dans les variables `jointure_inner`, `jointure_left` et `jointure_outer`. Affiche le nombre de lignes de chacune, puis la jointure `outer`.

Pourquoi la colonne `id_commande` de la jointure `outer` contient-elle des nombres décimaux ?

**Résultat attendu** :

```text
4 5 6
   id_commande code_client  montant     nom
0          1.0          C1    120.0  Durand
1          3.0          C1     40.0  Durand
2          2.0          C2     75.5  Martin
3          5.0          C3     15.0   Petit
4          4.0          C4    210.0     NaN
5          NaN          C5      NaN   Leroy
```

In [22]:
commandes = pd.DataFrame({
    "id_commande": [1, 2, 3, 4, 5],
    "code_client": ["C1", "C2", "C1", "C4", "C3"],
    "montant": [120.0, 75.5, 40.0, 210.0, 15.0],
})
clients = pd.DataFrame({
    "code_client": ["C1", "C2", "C3", "C5"],
    "nom": ["Durand", "Martin", "Petit", "Leroy"],
})

jointure_inner = commandes.merge(clients, on="code_client")
jointure_left = commandes.merge(clients, on="code_client", how="left")
jointure_outer = commandes.merge(clients, on="code_client", how="outer")
print(len(jointure_inner), len(jointure_left), len(jointure_outer))
jointure_outer


4 5 6


,id_commande,code_client,montant,nom
0,1.0,C1,120.0,Durand
1,3.0,C1,40.0,Durand
2,2.0,C2,75.5,Martin
3,5.0,C3,15.0,Petit
4,4.0,C4,210.0,NaN
5,NaN,C5,NaN,Leroy


In [23]:
# Vérification
assert len(jointure_inner) == 4, "inner ne garde que les clés présentes des deux côtés : la commande du client C4 disparaît"
assert len(jointure_left) == 5, "left garde toutes les commandes"
assert len(jointure_outer) == 6, "outer garde toutes les lignes des deux tables"
assert jointure_left.loc[jointure_left["code_client"] == "C4", "nom"].isna().all(), "dans la jointure left, le client C4 n'a pas de nom"
print("Question 2.1 : OK")


Question 2.1 : OK


**Réponse** : `inner` garde les 4 commandes dont le client existe ; `left` garde les 5 commandes, avec un nom manquant pour le client `C4` ; `outer` ajoute une sixième ligne pour le client `C5`, qui n'a aucune commande. Sur cette ligne, `id_commande` est manquant : une colonne d'entiers ne peut pas contenir `NaN`, pandas la convertit donc en nombres décimaux.

### Question 2.2

Refais la jointure `outer` avec `indicator=True`, dans une variable `controle_clients`. Affiche la répartition de la colonne `_merge`, puis construis et affiche :

- `codes_inconnus` : la liste des codes clients des commandes dont le client est inconnu (`left_only`) ;
- `clients_sans_commande` : la liste des codes des clients sans commande (`right_only`).

**Résultat attendu** :

```text
_merge
both          4
left_only     1
right_only    1
Name: count, dtype: int64
['C4']
['C5']
```

In [24]:
controle_clients = commandes.merge(clients, on="code_client", how="outer", indicator=True)
print(controle_clients["_merge"].value_counts())

codes_inconnus = list(controle_clients.loc[controle_clients["_merge"] == "left_only", "code_client"])
clients_sans_commande = list(controle_clients.loc[controle_clients["_merge"] == "right_only", "code_client"])
print(codes_inconnus)
print(clients_sans_commande)


_merge
both          4
left_only     1
right_only    1
Name: count, dtype: int64
['C4']
['C5']


In [25]:
# Vérification
assert "_merge" in controle_clients.columns, "ajoute indicator=True à la jointure"
assert list(codes_inconnus) == ["C4"], "seul le client C4 est inconnu du fichier clients"
assert list(clients_sans_commande) == ["C5"], "seul le client C5 n'a passé aucune commande"
print("Question 2.2 : OK")


Question 2.2 : OK


### Question 2.3

Dans `clients_doublon`, le client `C1` apparaît deux fois.

1. Joins `commandes` à `clients_doublon` en jointure `left`, dans `jointure_doublon`. Affiche son nombre de lignes et le total de sa colonne `montant`, et compare-les à ceux de `commandes`.
2. Essaie la même jointure en ajoutant `validate="many_to_one"` : pandas s'arrête sur une `MergeError`. Lis le message.
3. Supprime le doublon avec `drop_duplicates(subset="code_client")`, dans une table `clients_corriges`, puis refais la jointure avec `validate="many_to_one"`, dans `jointure_validee`. Affiche son nombre de lignes et son total.

Ne garde dans la cellule que les instructions qui fonctionnent, pour que le notebook s'exécute de haut en bas sans erreur.

**Résultat attendu** :

```text
5 460.5
7 620.5
5 460.5
```

In [26]:
clients_doublon = pd.DataFrame({
    "code_client": ["C1", "C2", "C3", "C1"],
    "nom": ["Durand", "Martin", "Petit", "Durand SA"],
})

jointure_doublon = commandes.merge(clients_doublon, on="code_client", how="left")
print(len(commandes), commandes["montant"].sum())
print(len(jointure_doublon), jointure_doublon["montant"].sum())

clients_corriges = clients_doublon.drop_duplicates(subset="code_client")
jointure_validee = commandes.merge(clients_corriges, on="code_client", how="left", validate="many_to_one")
print(len(jointure_validee), jointure_validee["montant"].sum())


5 460.5
7 620.5
5 460.5


In [27]:
# Vérification
assert len(jointure_doublon) == 7, "chaque commande du client C1 est dupliquée : 7 lignes"
assert clients_corriges["code_client"].is_unique, "clients_corriges ne doit plus contenir de doublon"
assert len(jointure_validee) == 5 and jointure_validee["montant"].sum() == 460.5, "après correction, on retrouve les 5 commandes et leur total"
print("Question 2.3 : OK")


Question 2.3 : OK


**Réponse** : Les deux commandes du client `C1` sont dupliquées : 7 lignes au lieu de 5, et un total de 620,50 euros au lieu de 460,50, sans aucun message. Avec `validate`, ce piège silencieux devient une erreur explicite. `drop_duplicates` garde la première occurrence : dans un vrai projet, il faudrait d'abord savoir laquelle des deux fiches du client est la bonne.

### Question 2.4

Dans `livraisons`, la clé s'appelle `client` et non `code_client`. Les deux tables ont aussi une colonne `ville` : la ville de livraison d'un côté, la ville du client de l'autre.

1. Joins `livraisons` et `clients_villes` avec `left_on` et `right_on`, et affiche seulement la liste des colonnes du résultat. Que sont devenues les colonnes `ville` ?
2. Refais la jointure avec `suffixes=("_livraison", "_client")`, dans `jointure_villes`, et affiche-la.
3. Compte les commandes livrées dans une autre ville que celle du client.

**Résultat attendu** :

```text
['id_commande', 'client', 'ville_x', 'code_client', 'ville_y']
   id_commande client ville_livraison code_client ville_client
0            1     C1            Lyon          C1         Lyon
1            2     C2           Paris          C2        Lille
2            3     C3          Rennes          C3       Rennes
1
```

In [28]:
livraisons = pd.DataFrame({
    "id_commande": [1, 2, 3],
    "client": ["C1", "C2", "C3"],
    "ville": ["Lyon", "Paris", "Rennes"],
})
clients_villes = pd.DataFrame({
    "code_client": ["C1", "C2", "C3"],
    "ville": ["Lyon", "Lille", "Rennes"],
})

print(list(livraisons.merge(clients_villes, left_on="client", right_on="code_client").columns))

jointure_villes = livraisons.merge(
    clients_villes, left_on="client", right_on="code_client", suffixes=("_livraison", "_client")
)
print(jointure_villes)
print((jointure_villes["ville_livraison"] != jointure_villes["ville_client"]).sum())


['id_commande', 'client', 'ville_x', 'code_client', 'ville_y']
   id_commande client ville_livraison code_client ville_client
0            1     C1            Lyon          C1         Lyon
1            2     C2           Paris          C2        Lille
2            3     C3          Rennes          C3       Rennes
1


In [29]:
# Vérification
assert {"ville_livraison", "ville_client"} <= set(jointure_villes.columns), "utilise suffixes=(\"_livraison\", \"_client\")"
assert len(jointure_villes) == 3, "chaque livraison trouve son client"
print("Question 2.4 : OK")


Question 2.4 : OK


**Réponse** : Les deux colonnes `ville` ont le même nom en dehors de la clé : pandas les renomme `ville_x` et `ville_y`, des noms peu parlants. Avec `suffixes`, on obtient `ville_livraison` et `ville_client`. Avec `left_on` et `right_on`, les deux colonnes clés `client` et `code_client` sont conservées. Une seule commande, la 2, est livrée ailleurs que dans la ville du client.

### Question 2.5

Passons aux vraies données. Charge `produits.csv` et `regions.csv` dans les variables `produits` et `regions`. Affiche leurs formes, puis vérifie que leurs clés (`id_produit` et `code_region`) sont bien uniques.

**Résultat attendu** :

```text
(16, 4) (6, 3)
True True
```

In [30]:
produits = pd.read_csv(DOSSIER / "produits.csv")
regions = pd.read_csv(DOSSIER / "regions.csv")

print(produits.shape, regions.shape)
print(produits["id_produit"].is_unique, regions["code_region"].is_unique)


(16, 4) (6, 3)
True True


In [31]:
# Vérification
assert produits.shape == (16, 4), "le catalogue contient 16 produits et 4 colonnes"
assert regions.shape == (6, 3), "la table des régions contient 6 régions et 3 colonnes"
print("Question 2.5 : OK")


Question 2.5 : OK


### Question 2.6

Joins les ventes au catalogue des produits, une première fois avec la jointure par défaut, une seconde fois avec `how="left"`. Range le nombre de lignes de chaque résultat dans `nb_inner` et `nb_left`, et compare-les au nombre de ventes. Que s'est-il passé ?

**Résultat attendu** :

```text
ventes : 2182
inner  : 2165
left   : 2182
```

In [32]:
nb_inner = len(ventes.merge(produits, on="id_produit"))
nb_left = len(ventes.merge(produits, on="id_produit", how="left"))
print("ventes :", len(ventes))
print("inner  :", nb_inner)
print("left   :", nb_left)


ventes : 2182
inner  : 2165
left   : 2182


In [33]:
# Vérification
assert nb_inner == 2165, "la jointure par défaut (inner) perd des ventes"
assert nb_left == 2182, "la jointure left garde toutes les ventes"
print("Question 2.6 : OK")


Question 2.6 : OK


**Réponse** : La jointure par défaut, `inner`, a fait disparaître 17 ventes sans aucun message : leur produit n'existe pas dans le catalogue. La jointure `left` garde toutes les ventes.

### Question 2.7

Refais la jointure `left` avec `indicator=True`, dans une variable `controle`, et affiche la répartition de la colonne `_merge`. Range ensuite dans `hors_catalogue` les ventes sans correspondance, et affiche : le produit concerné, le nombre de ventes, leur montant total (prix unitaire multiplié par la quantité, arrondi à 2 décimales) et la date de la première de ces ventes.

**Résultat attendu** :

```text
_merge
both          2165
left_only       17
right_only       0
Name: count, dtype: int64
produit : ['P17']
ventes  : 17
montant : 2673.62
depuis  : 2025-08-08 00:00:00
```

In [34]:
controle = ventes.merge(produits, on="id_produit", how="left", indicator=True)
print(controle["_merge"].value_counts())

hors_catalogue = controle[controle["_merge"] == "left_only"]
print("produit :", list(hors_catalogue["id_produit"].unique()))
print("ventes  :", len(hors_catalogue))
print("montant :", round((hors_catalogue["prix_unitaire"] * hors_catalogue["quantite"]).sum(), 2))
print("depuis  :", hors_catalogue["date"].min())


_merge
both          2165
left_only       17
right_only       0
Name: count, dtype: int64
produit : ['P17']
ventes  : 17
montant : 2673.62
depuis  : 2025-08-08 00:00:00


In [35]:
# Vérification
assert len(controle) == 2182 and "_merge" in controle.columns, "controle : jointure left avec indicator=True"
assert len(hors_catalogue) == 17, "17 ventes n'ont pas de correspondance dans le catalogue"
assert set(hors_catalogue["id_produit"]) == {"P17"}, "un seul produit est concerné"
print("Question 2.7 : OK")


Question 2.7 : OK


**Réponse** : Le produit `P17` est vendu depuis le 8 août sans exister dans le catalogue : 17 ventes, pour 2 673,62 euros. C'est une information à remonter au service qui gère le catalogue.

### Question 2.8

Construis la table `df` : les ventes jointes d'abord aux produits, puis aux régions, en jointure `left` et avec `validate="many_to_one"`. Affiche sa forme et vérifie qu'aucune vente n'est restée sans région.

**Résultat attendu** :

```text
(2182, 13)
0
```

In [36]:
df = (
    ventes
    .merge(produits, on="id_produit", how="left", validate="many_to_one")
    .merge(regions, on="code_region", how="left", validate="many_to_one")
)
print(df.shape)
print(df["region"].isna().sum())


(2182, 13)
0


In [37]:
# Vérification
assert df.shape == (2182, 13), "df doit garder les 2182 ventes, avec 13 colonnes"
assert list(df.columns[-5:]) == ["produit", "categorie", "prix_catalogue", "region", "objectif_mensuel"], "joins d'abord les produits, puis les régions"
assert df["objectif_mensuel"].notna().all(), "chaque vente doit avoir trouvé sa région"
print("Question 2.8 : OK")


Question 2.8 : OK


### Question 2.9

Les ventes du produit hors catalogue n'ont ni nom ni catégorie. Dans `df`, remplace ces valeurs manquantes des colonnes `produit` et `categorie` par le libellé `Non référencé`, pour qu'elles restent visibles dans les agrégations. Explique en une phrase pourquoi il vaut mieux ne pas supprimer ces ventes.

In [38]:
df[["produit", "categorie"]] = df[["produit", "categorie"]].fillna("Non référencé")
df[["produit", "categorie"]].isna().sum()


produit      0
categorie    0
dtype: int64

In [39]:
# Vérification
assert df[["produit", "categorie"]].isna().sum().sum() == 0, "il ne doit plus manquer de nom ni de catégorie"
assert (df["categorie"] == "Non référencé").sum() == 17, "les 17 ventes de P17 doivent être classées Non référencé"
assert df["prix_catalogue"].isna().sum() == 17, "le prix catalogue de P17 reste inconnu : ne le remplace pas"
print("Question 2.9 : OK")


Question 2.9 : OK


**Réponse** : Ces ventes ont bien eu lieu : les supprimer fausserait le chiffre d'affaires. Avec un libellé explicite, elles restent visibles dans tous les indicateurs et signalent qu'il faut compléter le catalogue.

### Pour aller plus loin 2.10

Obtiens avec `join` la même table que `ventes.merge(produits, on="id_produit", how="left")`, dans une variable `avec_join`, et vérifie avec `equals` que les deux tables sont identiques.

**Résultat attendu** :

```text
True
```

In [40]:
avec_join = ventes.join(produits.set_index("id_produit"), on="id_produit")
avec_join.equals(ventes.merge(produits, on="id_produit", how="left"))


True

In [41]:
# Vérification
assert avec_join.equals(ventes.merge(produits, on="id_produit", how="left")), "join utilise l'index de la table de droite : set_index(\"id_produit\")"
print("Question 2.10 : OK")


Question 2.10 : OK


## Partie 3 : agréger

*À faire après la section 4 du cours : agréger.*

On travaille désormais sur `df`, la table des ventes enrichies.

### Question 3.1

Ajoute à `df` une colonne `montant`, égale au prix unitaire payé multiplié par la quantité, puis affiche le chiffre d'affaires total, arrondi à 2 décimales.

**Résultat attendu** :

```text
409501.59
```

In [42]:
df["montant"] = df["prix_unitaire"] * df["quantite"]
print(round(df["montant"].sum(), 2))


409501.59


In [43]:
# Vérification
assert round(df["montant"].sum(), 2) == 409501.59, "vérifie le calcul : prix_unitaire multiplié par quantite"
print("Question 3.1 : OK")


Question 3.1 : OK


### Question 3.2

Affiche, chacun en une ligne de code :

1. le panier moyen (montant moyen d'une vente), arrondi à 2 décimales ;
2. le montant médian, arrondi à 2 décimales ;
3. le nombre de commandes ;
4. le nombre de produits différents vendus ;
5. le nombre de régions.

Pourquoi le panier moyen est-il bien plus élevé que le montant médian ? Pourquoi trouve-t-on 17 produits différents, alors que le catalogue en compte 16 ?

**Résultat attendu** :

```text
187.67
104.7
2182
17
6
```

In [44]:
print(round(df["montant"].mean(), 2))
print(round(df["montant"].median(), 2))
print(df["id_commande"].count())
print(df["id_produit"].nunique())
print(df["region"].nunique())


187.67
104.7
2182
17
6


**Réponse** : Quelques grosses ventes, de mobilier ou d'écrans, tirent la moyenne vers le haut : la moitié des ventes font moins de 104,70 euros, mais la moyenne atteint 187,67 euros. Le dix-septième produit est `P17`, vendu sans être référencé au catalogue.

### Question 3.3

Calcule le chiffre d'affaires par région, trié du plus grand au plus petit, dans une variable `ca_region`. Affiche-le arrondi à 2 décimales, puis vérifie que la somme des régions est égale au chiffre d'affaires total.

**Résultat attendu** :

```text
region
Île-de-France           127936.94
Nouvelle-Aquitaine       71843.18
Auvergne-Rhône-Alpes     64856.25
Hauts-de-France          53467.33
Occitanie                50318.18
Bretagne                 41079.71
Name: montant, dtype: float64
True
```

In [45]:
ca_region = df.groupby("region")["montant"].sum().sort_values(ascending=False)
print(ca_region.round(2))
print(round(ca_region.sum(), 2) == round(df["montant"].sum(), 2))


region
Île-de-France           127936.94
Nouvelle-Aquitaine       71843.18
Auvergne-Rhône-Alpes     64856.25
Hauts-de-France          53467.33
Occitanie                50318.18
Bretagne                 41079.71
Name: montant, dtype: float64
True


In [46]:
# Vérification
assert len(ca_region) == 6, "une valeur par région"
assert ca_region.index[0] == "Île-de-France" and ca_region.is_monotonic_decreasing, "trie du plus grand au plus petit"
assert round(ca_region.sum(), 2) == 409501.59, "la somme des régions doit égaler le chiffre d'affaires total"
print("Question 3.3 : OK")


Question 3.3 : OK


### Question 3.4

Calcule le chiffre d'affaires par région et par catégorie, avec les clés en colonnes (`as_index=False`), dans un DataFrame `ca_region_categorie`. Affiche sa forme, puis ses lignes concernant la Bretagne, arrondies à 2 décimales. Pourquoi n'obtient-on pas 6 × 5 = 30 lignes ?

**Résultat attendu** :

```text
(29, 3)
     region       categorie   montant
5  Bretagne          Bureau  14278.54
6  Bretagne    Informatique  16817.03
7  Bretagne       Papeterie   6201.30
8  Bretagne  Électroménager   3782.84
```

In [47]:
ca_region_categorie = df.groupby(["region", "categorie"], as_index=False)["montant"].sum()
print(ca_region_categorie.shape)
ca_region_categorie[ca_region_categorie["region"] == "Bretagne"].round(2)


(29, 3)


,region,categorie,montant
5,Bretagne,Bureau,14278.54
6,Bretagne,Informatique,16817.03
7,Bretagne,Papeterie,6201.30
8,Bretagne,Électroménager,3782.84


In [48]:
# Vérification
assert set(ca_region_categorie.columns) == {"region", "categorie", "montant"}, "les clés doivent être des colonnes : as_index=False"
assert ca_region_categorie.shape == (29, 3), "29 combinaisons de région et de catégorie"
print("Question 3.4 : OK")


Question 3.4 : OK


**Réponse** : `groupby` ne crée que les combinaisons présentes dans les données. Le produit non référencé n'a été vendu nulle part en Bretagne : la combinaison Bretagne et `Non référencé` n'existe pas.

### Question 3.5

La table `controle` de la question 2.7 contient encore les ventes de `P17` sans catégorie. Ajoute-lui une colonne `montant` (sans toucher à `df`), puis calcule le chiffre d'affaires par catégorie de deux façons : avec `groupby("categorie")`, dans `sans_nan`, puis avec `groupby("categorie", dropna=False)`, dans `avec_nan`. Affiche le total de chacun, puis `avec_nan` arrondi à 2 décimales. Qu'en conclus-tu ?

**Résultat attendu** :

```text
406827.97 409501.59
categorie
Bureau            124548.45
Informatique      185274.68
Papeterie          59180.80
Électroménager     37824.04
NaN                 2673.62
Name: montant, dtype: float64
```

In [49]:
controle["montant"] = controle["prix_unitaire"] * controle["quantite"]
sans_nan = controle.groupby("categorie")["montant"].sum()
avec_nan = controle.groupby("categorie", dropna=False)["montant"].sum()
print(round(sans_nan.sum(), 2), round(avec_nan.sum(), 2))
avec_nan.round(2)


406827.97 409501.59


categorie
Bureau            124548.45
Informatique      185274.68
Papeterie          59180.80
Électroménager     37824.04
NaN                 2673.62
Name: montant, dtype: float64

In [50]:
# Vérification
assert round(sans_nan.sum(), 2) == 406827.97, "sans_nan : groupby ignore les catégories manquantes"
assert round(avec_nan.sum(), 2) == 409501.59, "avec_nan : dropna=False garde les ventes sans catégorie"
print("Question 3.5 : OK")


Question 3.5 : OK


**Réponse** : Par défaut, `groupby` ignore les lignes dont la clé est manquante : les 17 ventes de `P17` disparaissent, soit 2 673,62 euros, sans aucun message. D'où deux réflexes : comparer le total des groupes au total général, et donner un libellé explicite aux clés manquantes, comme on l'a fait dans `df`.

### Question 3.6

Par catégorie, calcule en une seule instruction, avec une agrégation nommée, dans un DataFrame `par_categorie` : le chiffre d'affaires `ca`, le nombre de commandes `nb_commandes`, le panier moyen `panier_moyen` et la quantité totale vendue `quantite`. Affiche-le arrondi à 2 décimales.

**Résultat attendu** :

```text
                       ca  nb_commandes  panier_moyen  quantite
categorie
Bureau          124548.45           410        303.78      1235
Informatique    185274.68           793        233.64      2424
Non référencé     2673.62            17        157.27        50
Papeterie        59180.80           694         85.27      7225
Électroménager   37824.04           268        141.13       777
```

In [51]:
par_categorie = df.groupby("categorie").agg(
    ca=("montant", "sum"),
    nb_commandes=("id_commande", "count"),
    panier_moyen=("montant", "mean"),
    quantite=("quantite", "sum"),
)
par_categorie.round(2)


,ca,nb_commandes,panier_moyen,quantite
categorie,,,,
Bureau,124548.45,410,303.78,1235
Informatique,185274.68,793,233.64,2424
Non référencé,2673.62,17,157.27,50
Papeterie,59180.80,694,85.27,7225
Électroménager,37824.04,268,141.13,777


In [52]:
# Vérification
assert list(par_categorie.columns) == ["ca", "nb_commandes", "panier_moyen", "quantite"], "nomme les colonnes ca, nb_commandes, panier_moyen et quantite, dans cet ordre"
assert len(par_categorie) == 5, "une ligne par catégorie, Non référencé comprise"
assert par_categorie.loc["Papeterie", "quantite"] == 7225 and par_categorie["nb_commandes"].sum() == 2182
assert round(par_categorie["ca"].sum(), 2) == 409501.59
print("Question 3.6 : OK")


Question 3.6 : OK


### Question 3.7

Sur la petite table `mini` ci-dessous, qui contient des valeurs manquantes, calcule par région avec une agrégation nommée, dans `resume_mini` :

- `nb_lignes` : la fonction `size` sur la colonne `montant` ;
- `nb_montants` : la fonction `count` sur la colonne `montant` ;
- `nb_produits` : la fonction `nunique` sur la colonne `produit` ;
- `ca` : la somme des montants.

Explique la différence entre les trois premières colonnes.

**Résultat attendu** :

```text
        nb_lignes  nb_montants  nb_produits     ca
region
BRE             2            2            1  225.0
IDF             3            2            2  160.0
```

In [53]:
mini = pd.DataFrame({
    "region": ["IDF", "IDF", "IDF", "BRE", "BRE"],
    "produit": ["P01", "P02", "P01", "P03", None],
    "montant": [120.0, None, 40.0, 210.0, 15.0],
})

resume_mini = mini.groupby("region").agg(
    nb_lignes=("montant", "size"),
    nb_montants=("montant", "count"),
    nb_produits=("produit", "nunique"),
    ca=("montant", "sum"),
)
resume_mini


,nb_lignes,nb_montants,nb_produits,ca
region,,,,
BRE,2,2,1,225.0
IDF,3,2,2,160.0


In [54]:
# Vérification
colonnes = ["nb_lignes", "nb_montants", "nb_produits"]
assert list(resume_mini.loc["IDF", colonnes]) == [3, 2, 2], "IDF : 3 lignes, 2 montants renseignés, 2 produits différents"
assert list(resume_mini.loc["BRE", colonnes]) == [2, 2, 1], "BRE : 2 lignes, 2 montants, 1 seul produit renseigné"
print("Question 3.7 : OK")


Question 3.7 : OK


**Réponse** : `size` compte toutes les lignes du groupe, valeurs manquantes comprises. `count` ne compte que les valeurs renseignées : en IDF, un montant manque. `nunique` compte les valeurs distinctes renseignées : en IDF, `P01` apparaît deux fois, il n'y a que 2 produits différents ; en Bretagne, le produit manquant n'est pas compté. `sum` ignore les valeurs manquantes.

### Question 3.8

Ajoute à `df` une colonne `part_ca_region` : la part de chaque vente dans le chiffre d'affaires de sa région, calculée avec `transform`. Vérifie que les parts totalisent 1 dans chaque région (arrondis à 6 décimales), puis affiche les 3 ventes qui pèsent le plus dans leur région (`nlargest`), avec les colonnes `id_commande`, `region`, `produit`, `montant` et `part_ca_region`, arrondies à 3 décimales.

Pourquoi ces trois ventes sont-elles toutes en Bretagne ?

**Résultat attendu** :

```text
region
Auvergne-Rhône-Alpes    1.0
Bretagne                1.0
Hauts-de-France         1.0
Nouvelle-Aquitaine      1.0
Occitanie               1.0
Île-de-France           1.0
Name: part_ca_region, dtype: float64
      id_commande    region              produit  montant  part_ca_region
1557        11558  Bretagne  Bureau assis-debout   1995.0           0.049
826         10827  Bretagne  Bureau assis-debout   1795.5           0.044
411         10412  Bretagne  Bureau assis-debout   1596.0           0.039
```

In [55]:
df["part_ca_region"] = df["montant"] / df.groupby("region")["montant"].transform("sum")
print(df.groupby("region")["part_ca_region"].sum().round(6))
df.nlargest(3, "part_ca_region")[["id_commande", "region", "produit", "montant", "part_ca_region"]].round(3)


region
Auvergne-Rhône-Alpes    1.0
Bretagne                1.0
Hauts-de-France         1.0
Nouvelle-Aquitaine      1.0
Occitanie               1.0
Île-de-France           1.0
Name: part_ca_region, dtype: float64


,id_commande,region,produit,montant,part_ca_region
1557,11558,Bretagne,Bureau assis-debout,1995.0,0.049
826,10827,Bretagne,Bureau assis-debout,1795.5,0.044
411,10412,Bretagne,Bureau assis-debout,1596.0,0.039


In [56]:
# Vérification
assert len(df) == 2182, "transform renvoie une valeur par ligne : df garde ses 2182 lignes"
assert (df.groupby("region")["part_ca_region"].sum() - 1).abs().max() < 1e-9, "les parts de chaque région doivent totaliser 1"
assert round(df.loc[df["id_commande"] == 11558, "part_ca_region"].iloc[0], 4) == 0.0486, "divise chaque montant par le chiffre d'affaires de sa région"
print("Question 3.8 : OK")


Question 3.8 : OK


**Réponse** : La Bretagne a le plus petit chiffre d'affaires des six régions : une grosse vente, comme un bureau assis-debout à 1 995 euros, y pèse près de 5 % du total, plus que n'importe quelle vente dans une grande région.

### Question 3.9

Quels sont les 5 produits qui rapportent le plus de chiffre d'affaires ? Range-les dans `top_ca`. Et les 5 produits les plus vendus en quantité ? Range-les dans `top_quantite`. Affiche les deux, puis explique pourquoi ces classements sont si différents.

**Résultat attendu** :

```text
produit
Écran 24 pouces        66480.75
Chaise ergonomique     62088.15
Clavier mécanique      40695.25
Bureau assis-debout    38403.75
Casque audio           35064.15
Name: montant, dtype: float64
produit
Bloc-notes         2497
Classeur           2482
Agrafeuse          1323
Stylo plume         923
Souris sans fil     790
Name: quantite, dtype: int64
```

In [57]:
top_ca = df.groupby("produit")["montant"].sum().nlargest(5)
top_quantite = df.groupby("produit")["quantite"].sum().nlargest(5)
print(top_ca.round(2))
print(top_quantite)


produit
Écran 24 pouces        66480.75
Chaise ergonomique     62088.15
Clavier mécanique      40695.25
Bureau assis-debout    38403.75
Casque audio           35064.15
Name: montant, dtype: float64
produit
Bloc-notes         2497
Classeur           2482
Agrafeuse          1323
Stylo plume         923
Souris sans fil     790
Name: quantite, dtype: int64


In [58]:
# Vérification
assert list(top_ca.index) == ["Écran 24 pouces", "Chaise ergonomique", "Clavier mécanique", "Bureau assis-debout", "Casque audio"], "top_ca : chiffre d'affaires par produit, 5 plus grandes valeurs"
assert list(top_quantite.index[:2]) == ["Bloc-notes", "Classeur"] and len(top_quantite) == 5, "top_quantite : quantité totale par produit, 5 plus grandes valeurs"
print("Question 3.9 : OK")


Question 3.9 : OK


**Réponse** : La papeterie se vend en grande quantité, mais à quelques euros l'unité ; les écrans et le mobilier se vendent moins souvent, mais à plusieurs centaines d'euros. Le classement dépend de l'indicateur choisi : il faut toujours préciser lequel on présente.

### Question 3.10

Pour chaque région, trouve le produit qui rapporte le plus de chiffre d'affaires. Le résultat attendu est un DataFrame `meilleur_produit` avec les colonnes `region`, `produit` et `montant`, une ligne par région.

Méthode : calcule le chiffre d'affaires par région et par produit (`as_index=False`), trie par montant décroissant, puis garde la première ligne de chaque région avec `groupby("region").head(1)`.

**Résultat attendu** :

```text
                   region              produit   montant
100         Île-de-France      Écran 24 pouces  24267.60
56     Nouvelle-Aquitaine   Chaise ergonomique  16446.45
83              Occitanie      Écran 24 pouces   9402.75
6    Auvergne-Rhône-Alpes   Chaise ergonomique   9013.80
49        Hauts-de-France      Écran 24 pouces   8287.65
20               Bretagne  Bureau assis-debout   6583.50
```

In [59]:
ca_region_produit = df.groupby(["region", "produit"], as_index=False)["montant"].sum()
meilleur_produit = (
    ca_region_produit
    .sort_values("montant", ascending=False)
    .groupby("region")
    .head(1)
)
meilleur_produit.round(2)


,region,produit,montant
100,Île-de-France,Écran 24 pouces,24267.60
56,Nouvelle-Aquitaine,Chaise ergonomique,16446.45
83,Occitanie,Écran 24 pouces,9402.75
6,Auvergne-Rhône-Alpes,Chaise ergonomique,9013.80
49,Hauts-de-France,Écran 24 pouces,8287.65
20,Bretagne,Bureau assis-debout,6583.50


In [60]:
# Vérification
assert len(meilleur_produit) == 6, "une ligne par région"
assert dict(zip(meilleur_produit["region"], meilleur_produit["produit"])) == {
    "Île-de-France": "Écran 24 pouces",
    "Nouvelle-Aquitaine": "Chaise ergonomique",
    "Occitanie": "Écran 24 pouces",
    "Auvergne-Rhône-Alpes": "Chaise ergonomique",
    "Hauts-de-France": "Écran 24 pouces",
    "Bretagne": "Bureau assis-debout",
}, "trie par montant décroissant avant de garder la première ligne de chaque région"
print("Question 3.10 : OK")


Question 3.10 : OK


### Question 3.11

Ajoute à `df` une colonne `mois` avec `.dt.to_period("M")`. Construis ensuite avec `pivot_table`, dans `tableau_mois_region`, le tableau croisé du chiffre d'affaires, avec les mois en lignes, les codes de région (`code_region`) en colonnes et les totaux en marge. Affiche-le arrondi à l'euro, en entiers (`round(0)`, puis `astype(int)`).

**Résultat attendu** :

```text
code_region    ARA    BRE    HDF     IDF    NAQ    OCC     All
mois
2025-01       9608   3016   3656   15792   6525   4878   43475
2025-02       5118   4814   6575   10702   5033   5588   37830
2025-03      11593   2844   6888   15992  12123   4031   53472
2025-04       5933   7575   8321   16469  10258   7785   56340
2025-05       7614   4266   5178   16173   8450   7119   48802
2025-06       8287   5427   5814   16153  10644   6037   52363
2025-07       3686   5498   3465   10651   4146   4426   31873
2025-08       2081   1361   3363    9987   6483   3998   27274
2025-09      10935   6280  10207   16015   8181   6456   58074
All          64856  41080  53467  127937  71843  50318  409502
```

In [61]:
df["mois"] = df["date"].dt.to_period("M")

tableau_mois_region = pd.pivot_table(
    df,
    index="mois",
    columns="code_region",
    values="montant",
    aggfunc="sum",
    margins=True,
)
tableau_mois_region.round(0).astype(int)


code_region,ARA,BRE,HDF,IDF,NAQ,OCC,All
mois,,,,,,,
2025-01,9608,3016,3656,15792,6525,4878,43475
2025-02,5118,4814,6575,10702,5033,5588,37830
2025-03,11593,2844,6888,15992,12123,4031,53472
2025-04,5933,7575,8321,16469,10258,7785,56340
2025-05,7614,4266,5178,16173,8450,7119,48802
2025-06,8287,5427,5814,16153,10644,6037,52363
2025-07,3686,5498,3465,10651,4146,4426,31873
2025-08,2081,1361,3363,9987,6483,3998,27274
2025-09,10935,6280,10207,16015,8181,6456,58074


In [62]:
# Vérification
assert tableau_mois_region.shape == (10, 7), "9 mois et la ligne de total, 6 régions et la colonne de total : margins=True"
assert round(float(tableau_mois_region.loc["All", "All"])) == 409502, "le total général doit égaler le chiffre d'affaires total"
print("Question 3.11 : OK")


Question 3.11 : OK


### Question 3.12

Avec `crosstab`, compte le nombre de ventes par code de région et par fidélité du client, avec les totaux en marge. Calcule ensuite, dans `fidelite`, la proportion de ventes à des clients fidèles et non fidèles dans chaque région (`normalize="index"`), arrondie à 2 décimales. La fidélité est-elle homogène d'une région à l'autre ?

**Résultat attendu** :

```text
client_fidele  False  True   All
code_region
ARA              254   124   378
BRE              130    79   209
HDF              195   125   320
IDF              453   221   674
NAQ              207   106   313
OCC              184   104   288
All             1423   759  2182
client_fidele  False  True
code_region
ARA             0.67   0.33
BRE             0.62   0.38
HDF             0.61   0.39
IDF             0.67   0.33
NAQ             0.66   0.34
OCC             0.64   0.36
```

In [63]:
print(pd.crosstab(df["code_region"], df["client_fidele"], margins=True))
fidelite = pd.crosstab(df["code_region"], df["client_fidele"], normalize="index").round(2)
fidelite


client_fidele  False  True   All
code_region                     
ARA              254   124   378
BRE              130    79   209
HDF              195   125   320
IDF              453   221   674
NAQ              207   106   313
OCC              184   104   288
All             1423   759  2182


client_fidele,False,True
code_region,,
ARA,0.67,0.33
BRE,0.62,0.38
HDF,0.61,0.39
IDF,0.67,0.33
NAQ,0.66,0.34
OCC,0.64,0.36


In [64]:
# Vérification
assert fidelite.shape == (6, 2), "une ligne par région, une colonne par valeur de client_fidele"
assert ((fidelite.sum(axis=1) - 1).abs() <= 0.011).all(), "normalize=\"index\" : chaque ligne totalise 1"
print("Question 3.12 : OK")


Question 3.12 : OK


**Réponse** : Entre 33 % et 39 % des ventes sont faites à des clients fidèles selon la région : la fidélité est assez homogène.

### Pour aller plus loin 3.13

`crosstab` peut aussi agréger une colonne de valeurs, avec les paramètres `values` et `aggfunc`. Calcule dans `part_categorie` la part de chaque catégorie dans le chiffre d'affaires de chaque région : codes de région en lignes, catégories en colonnes, `values=df["montant"]`, `aggfunc="sum"` et `normalize="index"`. Affiche-la arrondie à 2 décimales. Dans quelle région l'informatique pèse-t-elle le plus ?

**Résultat attendu** :

```text
categorie    Bureau  Informatique  Non référencé  Papeterie  Électroménager
code_region
ARA            0.27          0.40           0.00       0.17            0.15
BRE            0.35          0.41           0.00       0.15            0.09
HDF            0.26          0.45           0.02       0.16            0.12
IDF            0.29          0.50           0.01       0.13            0.07
NAQ            0.39          0.42           0.00       0.13            0.06
OCC            0.27          0.48           0.01       0.14            0.10
```

In [65]:
part_categorie = pd.crosstab(
    df["code_region"], df["categorie"], values=df["montant"], aggfunc="sum", normalize="index"
)
part_categorie.round(2)


categorie,Bureau,Informatique,Non référencé,Papeterie,Électroménager
code_region,,,,,
ARA,0.27,0.40,0.00,0.17,0.15
BRE,0.35,0.41,0.00,0.15,0.09
HDF,0.26,0.45,0.02,0.16,0.12
IDF,0.29,0.50,0.01,0.13,0.07
NAQ,0.39,0.42,0.00,0.13,0.06
OCC,0.27,0.48,0.01,0.14,0.10


In [66]:
# Vérification
assert part_categorie.shape == (6, 5), "6 régions en lignes, 5 catégories en colonnes"
assert part_categorie["Informatique"].idxmax() == "IDF"
print("Question 3.13 : OK")


Question 3.13 : OK


**Réponse** : En Île-de-France, où l'informatique représente la moitié du chiffre d'affaires.

### Pour aller plus loin 3.14

Le prix payé (`prix_unitaire`) est parfois inférieur au prix catalogue. Sans ajouter de colonne à `df` :

1. calcule le taux de remise de chaque vente, `1 - prix_unitaire / prix_catalogue`, arrondi à 2 décimales, dans une Series `taux_remise`, et affiche sa répartition avec `value_counts(dropna=False)` ;
2. calcule, pour les clients fidèles et non fidèles, la proportion de ventes faites sous le prix catalogue (indice : `assign` crée une colonne booléenne temporaire, et la moyenne d'une colonne booléenne est une proportion).

Les clients fidèles bénéficient-ils plus souvent d'une remise ?

**Résultat attendu** :

```text
0.00    1312
0.05     451
0.10     402
NaN       17
Name: count, dtype: int64
client_fidele
False    0.383
True     0.406
Name: remisee, dtype: float64
```

In [67]:
taux_remise = (1 - df["prix_unitaire"] / df["prix_catalogue"]).round(2)
print(taux_remise.value_counts(dropna=False))

(
    df.assign(remisee=df["prix_unitaire"] < df["prix_catalogue"])
    .groupby("client_fidele")["remisee"]
    .mean()
    .round(3)
)


0.00    1312
0.05     451
0.10     402
NaN       17
Name: count, dtype: int64


client_fidele
False    0.383
True     0.406
Name: remisee, dtype: float64

**Réponse** : Trois niveaux sont pratiqués : pas de remise, 5 % et 10 %. Les 17 valeurs manquantes sont les ventes de `P17`, qui n'a pas de prix catalogue. Les clients fidèles ont une remise dans 40,6 % des ventes, contre 38,3 % pour les autres : la différence est faible.